# TFT Data Preparation

This notebook prepares the selected Store 3 + Item 589403 time series
for training a Temporal Fusion Transformer (TFT).

Selected series:
- Store: 3
- Item: 589403

In [15]:
# Pandas will be used for loading and preparing our time-series data.
import pandas as pd


# Load the final store-item series that we selected in Notebook 02.
# We saved this smaller CSV so we no longer need to scan the original 5 GB train.csv.
series_df = pd.read_csv(
    "../data/favorita/store3_item589403.csv"
)


# Convert 'date' from a string into Pandas datetime format.
# This allows us to sort dates, calculate date ranges, extract weekdays, etc.
series_df["date"] = pd.to_datetime(series_df["date"])


# Display the first few rows to confirm that the file loaded correctly.
series_df.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,3285,2013-01-02,3,589403,1.0,NaN
1,44830,2013-01-03,3,589403,3.0,NaN
2,84937,2013-01-04,3,589403,3.0,NaN
3,125113,2013-01-05,3,589403,3.0,NaN
4,167310,2013-01-06,3,589403,2.0,NaN


In [16]:
# Create a complete sequence of daily dates from the
# first date in our dataset to the last date.
full_date_range = pd.date_range(
    start=series_df["date"].min(),
    end=series_df["date"].max(),
    freq="D"  # "D" means daily frequency
)


# Create a DataFrame containing every calendar date.
# This should contain 1,687 rows.
daily_df = pd.DataFrame({
    "date": full_date_range
})


# Merge our original sales data onto the complete calendar.
#
# "how='left'" means:
# Keep EVERY date from daily_df, even when that date
# does not exist in our original series_df.
daily_df = daily_df.merge(
    series_df,
    on="date",
    how="left"
)


# Check the result.
print("Original rows:", len(series_df))
print("Continuous calendar rows:", len(daily_df))
print("Missing sales values:", daily_df["unit_sales"].isna().sum())

daily_df.head()

Original rows: 1593
Continuous calendar rows: 1687
Missing sales values: 94


,date,id,store_nbr,item_nbr,unit_sales,onpromotion
0,2013-01-02,3285.0,3.0,589403.0,1.0,NaN
1,2013-01-03,44830.0,3.0,589403.0,3.0,NaN
2,2013-01-04,84937.0,3.0,589403.0,3.0,NaN
3,2013-01-05,125113.0,3.0,589403.0,3.0,NaN
4,2013-01-06,167310.0,3.0,589403.0,2.0,NaN


In [17]:
# Fill the 94 newly created missing sales values with 0.
#
# These dates did not have a store-item sales record in the raw
# Favorita training data, so we represent them as zero recorded sales.
daily_df["unit_sales"] = daily_df["unit_sales"].fillna(0)


# Verify that our target variable now has no missing values.
print(
    "Missing unit_sales after filling:",
    daily_df["unit_sales"].isna().sum()
)


# Count how many days now have zero recorded sales.
print(
    "Zero-sales days:",
    (daily_df["unit_sales"] == 0).sum()
)

Missing unit_sales after filling: 0
Zero-sales days: 94


In [18]:
# Every row in this dataframe represents the same store-item series.
# The newly created calendar dates currently have NaN for these columns,
# so we explicitly restore the correct store and item numbers.
daily_df["store_nbr"] = 3
daily_df["item_nbr"] = 589403


# Convert them back to integers.
# The merge changed them to floats because columns containing NaN
# are often represented as floating-point numbers by Pandas.
daily_df["store_nbr"] = daily_df["store_nbr"].astype(int)
daily_df["item_nbr"] = daily_df["item_nbr"].astype(int)


# Verify that there are no missing store/item values.
print("Missing store numbers:", daily_df["store_nbr"].isna().sum())
print("Missing item numbers:", daily_df["item_nbr"].isna().sum())

Missing store numbers: 0
Missing item numbers: 0


In [19]:
# Create a cleaner categorical promotion variable.
#
# True  = we know the item was on promotion
# False = we know the item was not on promotion
#
# Missing values are not mapped here, so they temporarily remain NaN.
daily_df["promotion_status"] = daily_df["onpromotion"].map({
    True: "promotion",
    False: "no_promotion"
})


# Replace missing promotion information with an explicit "unknown" category.
# This is better than incorrectly assuming that missing means no promotion.
daily_df["promotion_status"] = (
    daily_df["promotion_status"].fillna("unknown")
)


# Check how many days belong to each category.
print(daily_df["promotion_status"].value_counts())

promotion_status
no_promotion    739
unknown         532
promotion       416
Name: count, dtype: int64


In [20]:
# Create a sequential time index.
# The first day gets 0, the second day gets 1, and so on.
# TFT uses this to understand the ordering of observations through time.
daily_df["time_idx"] = range(len(daily_df))


# Extract the weekday from each date.
# Example:
# 2013-01-02 -> Wednesday
daily_df["day_of_week"] = daily_df["date"].dt.day_name()


# Extract the month.
# Convert it to string because we will treat month as a category
# rather than as a continuous numerical measurement.
daily_df["month"] = daily_df["date"].dt.month.astype(str)


# Extract the year from the date.
daily_df["year"] = daily_df["date"].dt.year


# Extract the day number within the month.
# Example: January 17 -> 17
daily_df["day_of_month"] = daily_df["date"].dt.day


# Create an identifier for this entire time series.
# Every row belongs to Store 3 + Item 589403, so every row
# receives the same series ID.
daily_df["series_id"] = "store3_item589403"


# Display the features we just created.
daily_df[
    [
        "date",
        "unit_sales",
        "time_idx",
        "day_of_week",
        "month",
        "year",
        "day_of_month",
        "promotion_status",
        "series_id"
    ]
].head(10)

,date,unit_sales,time_idx,day_of_week,month,year,day_of_month,promotion_status,series_id
0,2013-01-02,1.0,0,Wednesday,1,2013,2,unknown,store3_item589403
1,2013-01-03,3.0,1,Thursday,1,2013,3,unknown,store3_item589403
2,2013-01-04,3.0,2,Friday,1,2013,4,unknown,store3_item589403
3,2013-01-05,3.0,3,Saturday,1,2013,5,unknown,store3_item589403
4,2013-01-06,2.0,4,Sunday,1,2013,6,unknown,store3_item589403
5,2013-01-07,5.0,5,Monday,1,2013,7,unknown,store3_item589403
6,2013-01-08,6.0,6,Tuesday,1,2013,8,unknown,store3_item589403
7,2013-01-09,3.0,7,Wednesday,1,2013,9,unknown,store3_item589403
8,2013-01-10,2.0,8,Thursday,1,2013,10,unknown,store3_item589403
9,2013-01-11,3.0,9,Friday,1,2013,11,unknown,store3_item589403


In [21]:
# Remove columns that we no longer need for TFT preparation.
#
# 'id' is only the original row identifier from Favorita.
# 'onpromotion' has been replaced by our cleaner
# 'promotion_status' categorical feature.
daily_df = daily_df.drop(
    columns=["id", "onpromotion"]
)


# Display the final column names after cleanup.
print(daily_df.columns.tolist())

['date', 'store_nbr', 'item_nbr', 'unit_sales', 'promotion_status', 'time_idx', 'day_of_week', 'month', 'year', 'day_of_month', 'series_id']


In [22]:
# TFT will look at up to 120 previous days of historical data
# when it tries to predict future demand.
#
# We increased this from 60 to 120 days because the model will now
# forecast much farther into the future. This gives TFT roughly
# four months of historical demand, seasonality, and promotion patterns.
max_encoder_length = 120


# TFT will predict up to the following 60 days of future demand.
#
# The final application can still offer shorter forecast horizons
# such as 7, 14, or 30 days by returning only the required portion
# of the full 60-day forecast.
max_prediction_length = 60


# Print the values so we can confirm our updated forecasting setup.
print("Historical window:", max_encoder_length, "days")
print("Maximum forecast horizon:", max_prediction_length, "days")

Historical window: 120 days
Maximum forecast horizon: 60 days


In [23]:
# Use 60 days for our final untouched test period.
test_length = max_prediction_length

# Use another 60 days immediately before the test period for validation.
validation_length = max_prediction_length


# LAST 60 days -> test set
test_df = daily_df.iloc[
    -test_length:
].copy()


# 60 days immediately before the test period -> validation set
validation_df = daily_df.iloc[
    -(test_length + validation_length):-test_length
].copy()


# Everything before those final 120 days -> training set
train_df = daily_df.iloc[
    :-(test_length + validation_length)
].copy()


# Print sizes so we can verify the split.
print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))


# Print date ranges so we can verify where each split starts and ends.
print(
    "\nTraining period:",
    train_df["date"].min(),
    "to",
    train_df["date"].max()
)

print(
    "Validation period:",
    validation_df["date"].min(),
    "to",
    validation_df["date"].max()
)

print(
    "Test period:",
    test_df["date"].min(),
    "to",
    test_df["date"].max()
)

Training rows: 1567
Validation rows: 60
Test rows: 60

Training period: 2013-01-02 00:00:00 to 2017-04-17 00:00:00
Validation period: 2017-04-18 00:00:00 to 2017-06-16 00:00:00
Test period: 2017-06-17 00:00:00 to 2017-08-15 00:00:00


Static categorical:
- series_id

Known future categorical:
- day_of_week
- month
- promotion_status   ← because we want promotion-plan forecasting

Known future real/numeric:
- time_idx
- year
- day_of_month

Unknown future real:
- unit_sales         ← target we are predicting

In [24]:
# Columns that stay constant for the entire time series.
# Every row belongs to the same store-item series.
static_categoricals = [
    "series_id"
]


# Categorical variables whose future values are known before forecasting.
#
# day_of_week and month can always be calculated from future dates.
#
# promotion_status is included here because our final application
# will allow the user to provide the planned promotion schedule
# for the next 14 days.
time_varying_known_categoricals = [
    "day_of_week",
    "month",
    "promotion_status"
]


# Numerical variables whose future values are known beforehand.
#
# time_idx simply increases through time.
# year and day_of_month can be calculated from the future date.
time_varying_known_reals = [
    "time_idx",
    "year",
    "day_of_month"
]


# Numerical variables that are NOT known in the future.
#
# unit_sales is our forecasting target:
# historical values are available, future values must be predicted.
time_varying_unknown_reals = [
    "unit_sales"
]


# Print everything so we can verify our setup.
print("Static categoricals:", static_categoricals)
print("Known categoricals:", time_varying_known_categoricals)
print("Known reals:", time_varying_known_reals)
print("Unknown reals:", time_varying_unknown_reals)

Static categoricals: ['series_id']
Known categoricals: ['day_of_week', 'month', 'promotion_status']
Known reals: ['time_idx', 'year', 'day_of_month']
Unknown reals: ['unit_sales']


In [25]:
# Import PyTorch.
# PyTorch provides the underlying deep-learning functionality
# that our TFT model will eventually use.
import torch


# TimeSeriesDataSet comes from PyTorch Forecasting.
# It converts our normal dataframe into time-series samples
# that models such as TFT can understand.
from pytorch_forecasting import TimeSeriesDataSet


# Print the versions so we know the libraries loaded successfully.
print("PyTorch version:", torch.__version__)

import pytorch_forecasting
print("PyTorch Forecasting version:", pytorch_forecasting.__version__)

PyTorch version: 2.14.0+cpu
PyTorch Forecasting version: 1.8.0


In [26]:
# sys gives us information about the Python environment
# currently running this Jupyter notebook.
import sys


# This prints the exact Python executable being used.
# We want it to point to our project's .venv folder.
print(sys.executable)

c:\Users\karra\Desktop\inventory-forecasting\.venv\Scripts\python.exe


In [27]:
# Install PyTorch and PyTorch Forecasting into the SAME Python
# environment that is currently running this notebook.
import sys

!{sys.executable} -m pip install torch pytorch-forecasting

'DOSKEY' is not recognized as an internal or external command,
operable program or batch file.

[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
# PyTorch Forecasting expects categorical variables to contain
# categorical/string-like values rather than numerical category codes.
#
# These three columns are categorical inputs to our TFT model.
categorical_columns = [
    "series_id",
    "day_of_week",
    "month",
    "promotion_status"
]

# Convert every categorical column explicitly to string.
# We do this in a loop so we don't have to repeat the same line four times.
for column in categorical_columns:
    daily_df[column] = daily_df[column].astype(str)


# Check the data types after conversion.
print(daily_df[categorical_columns].dtypes)

series_id           str
day_of_week         str
month               str
promotion_status    str
dtype: object


In [29]:
# The training cutoff is the final time index that belongs to training.
training_cutoff = train_df["time_idx"].max()

print("Training cutoff:", training_cutoff)

Training cutoff: 1566


In [30]:
# Create the dataset that PyTorch Forecasting will use for TFT training.
#
# We only provide rows up to training_cutoff.
# Therefore validation and test target periods are NOT included
# in the model's training data.
training_dataset = TimeSeriesDataSet(
    
    # Keep only the training portion of our complete daily dataframe.
    daily_df[daily_df["time_idx"] <= training_cutoff],

    # Column that tells TFT the chronological position of each row.
    time_idx="time_idx",

    # This is the value we ultimately want TFT to predict.
    target="unit_sales",

    # Identifies which rows belong to the same time series.
    # We currently have one series: Store 3 + Item 589403.
    group_ids=["series_id"],

    # TFT can look at up to 60 previous days when making a forecast.
    max_encoder_length=max_encoder_length,

    # TFT will learn to predict the next 14 days.
    max_prediction_length=max_prediction_length,

    # Information that stays constant for the whole series.
    static_categoricals=static_categoricals,

    # Categorical information known for future dates.
    # This includes our planned promotion status.
    time_varying_known_categoricals=time_varying_known_categoricals,

    # Numerical information known for future dates.
    time_varying_known_reals=time_varying_known_reals,

    # Numerical information NOT known in advance.
    # unit_sales is both our target and historical input.
    time_varying_unknown_reals=time_varying_unknown_reals,
)

In [31]:
# Check how many training samples/windows were created.
print("Number of training samples:", len(training_dataset))

Number of training samples: 1388


In [38]:
# Find the final time index belonging to the validation period.
validation_cutoff = validation_df["time_idx"].max()

# Find the final time index in the complete dataset.
final_time_index = daily_df["time_idx"].max()

print("Training cutoff:", training_cutoff)
print("Validation cutoff:", validation_cutoff)
print("Final time index:", final_time_index)

Training cutoff: 1566
Validation cutoff: 1626
Final time index: 1686


Validation samples: 1
Test samples: 1


In [39]:
# Create validation data using the same rules/configuration
# that we already defined for the training dataset.
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,

    # Include all historical data up to the end of validation.
    # This gives TFT the 120-day historical context it needs
    # before predicting the 60-day validation window.
    daily_df[daily_df["time_idx"] <= validation_cutoff],

    # Select the final 120 -> 60 forecasting window.
    predict=True,

    # Keep the validation window fixed rather than randomized.
    stop_randomization=True
)


# Create the test dataset using the same configuration.
test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,

    # The test period is at the very end of daily_df,
    # so we provide the complete dataframe.
    daily_df,

    # Select the final 120 -> 60 forecasting window.
    predict=True,

    # Keep the test window fixed.
    stop_randomization=True
)


# Check how many forecasting windows were created.
print("Validation samples:", len(validation_dataset))
print("Test samples:", len(test_dataset))

Validation samples: 1
Test samples: 1


In [40]:
# Get ONE batch from the training DataLoader.
#
# iter(...) lets us move through the DataLoader.
# next(...) asks for the very first batch.
x, y = next(iter(train_dataloader))


# x contains the INPUT information given to TFT.
# y contains the TARGET information that TFT should learn to predict.
print("Type of x:", type(x))
print("Type of y:", type(y))


# Show the different pieces of information contained inside x.
print("\nInput keys:")
print(x.keys())

Type of x: <class 'dict'>
Type of y: <class 'tuple'>

Input keys:
dict_keys(['encoder_cat', 'encoder_cont', 'encoder_target', 'encoder_lengths', 'decoder_cat', 'decoder_cont', 'decoder_target', 'decoder_lengths', 'decoder_time_idx', 'groups', 'target_scale'])


In [36]:
# Save the fully prepared daily dataframe.
#
# Notebook 04 can load this file directly instead of repeating
# all of the cleaning and feature-engineering work from Notebook 03.
daily_df.to_csv(
    "../data/favorita/store3_item589403_tft_ready.csv",
    index=False
)

print("Prepared TFT data saved successfully.")

Prepared TFT data saved successfully.
